In [1]:
# Progress Bar
import requests
exec(requests.get("https://raw.githubusercontent.com/razorrrzz/Test-1/main/ProgressBar/ProgressBar.py").text)

✅ Live Progress Bar & Timer Enabled!


In [2]:
# Dedicated index num and Dynamic 
import pandas as pd

# Save original HTML renderer once to prevent infinite loops
if not hasattr(pd.DataFrame, '_repr_html_orig'):
    pd.DataFrame._repr_html_orig = pd.DataFrame._repr_html_

def _dual_index_repr_html(self):
    temp_df = self.copy()
    
    # Push the dedicated index into a column if not already present
    if 'Original_Index' not in temp_df.columns:
        temp_df = temp_df.reset_index(names='Original_Index')
    
    # Apply fresh sequential 1-based index
    temp_df.index = range(1, len(temp_df) + 1)
    
    return temp_df._repr_html_orig()

# Apply universal display hook
pd.DataFrame._repr_html_ = _dual_index_repr_html

In [3]:
import pandas as pd

# Globally disable scientific notation and show standard decimals up to 8 places
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')

In [15]:
import aiohttp
import asyncio
import time
import pandas as pd

async def get_spot_and_coin_tickers():
    # Endpoints limited to Spot and COIN-M Futures
    endpoints = {
        "Spot": "https://api.binance.com/api/v3/ticker/price",
        }

    # ThreadedResolver prevents Android DNS failure in Pydroid 3
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())

    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [session.get(url) for url in endpoints.values()]
        
        start_time = time.time()
        responses = await asyncio.gather(*tasks)
        
        results = []
        for market, response in zip(endpoints.keys(), responses):
            if response.status == 200:
                data = await response.json()
                for item in data:
                    results.append({
                        "Market": market,
                        "Symbol": item.get("symbol"),
                        "Price": float(item.get("price")),
                    })
            else:
                print(f"Error fetching {market}: {response.status}")
                
        print(f"Successfully pulled {len(results)} tickers in {time.time() - start_time:.3f} seconds.")
        return results

# Run directly in a Jupyter Notebook cell
tickers_data = await get_spot_and_coin_tickers()

# Format into DataFrame
df = pd.DataFrame(tickers_data)
df.sort_values(by=["Market", "Symbol"], inplace=True)
df.reset_index(drop=True, inplace=True)

display(df)

Successfully pulled 3692 tickers in 0.994 seconds.


,Original_Index,Market,Symbol,Price
1,0,Spot,0GBNB,0.000863
2,1,Spot,0GFDUSD,0.837000
3,2,Spot,0GTRY,9.103000
4,3,Spot,0GUSDC,0.188500
5,4,Spot,0GUSDT,0.188700
...,...,...,...,...
3688,3687,Spot,币安人生TRY,3.600000
3689,3688,Spot,币安人生U,0.486400
3690,3689,Spot,币安人生USD1,0.487900
3691,3690,Spot,币安人生USDC,0.495200


In [16]:
# Filter for all pairs containing "BTC"
btc_pairs = df[df['Symbol'].str.contains('BTC')]
display(btc_pairs)

,Original_Index,Market,Symbol,Price
1,16,Spot,1INCHBTC,1.210000e-06
2,38,Spot,AAVEBTC,1.672000e-03
3,48,Spot,ABTC,1.680000e-06
4,49,Spot,ACABTC,3.200000e-07
5,54,Spot,ACEBTC,2.169000e-05
...,...,...,...,...
527,3650,Spot,ZENBTC,6.825000e-05
528,3657,Spot,ZILBTC,1.000000e-07
529,3664,Spot,ZKBTC,3.300000e-07
530,3677,Spot,ZROBTC,2.808000e-05


In [17]:
# Filter symbols where USDT or USDC is the quote/base currency
usdt_usdc_df = df[df['Symbol'].str.endswith(('USDT', 'USDC'))].copy()

# Alternatively, to catch any pair containing USDT or USDC anywhere in the name:
# usdt_usdc_df = df[df['Symbol'].str.contains('USDT|USDC')].copy()

usdt_usdc_df.reset_index(drop=True, inplace=True)

# Display total count and the filtered table
print(f"Total USDT/USDC pairs found: {len(usdt_usdc_df)}")
display(usdt_usdc_df)

Total USDT/USDC pairs found: 1074


,Original_Index,Market,Symbol,Price
1,0,Spot,0GUSDC,0.188500
2,1,Spot,0GUSDT,0.188700
3,2,Spot,1000CATUSDC,0.001969
4,3,Spot,1000CATUSDT,0.001964
5,4,Spot,1000CHEEMSUSDC,0.000551
...,...,...,...,...
1070,1069,Spot,ZROUSDC,1.110000
1071,1070,Spot,ZROUSDT,1.111000
1072,1071,Spot,ZRXUSDT,0.104000
1073,1072,Spot,币安人生USDC,0.495200


In [18]:
usdt_only = usdt_usdc_df[usdt_usdc_df['Symbol'].str.endswith('USDT')]
usdc_only = usdt_usdc_df[usdt_usdc_df['Symbol'].str.endswith('USDC')]

In [21]:
usdt_only

,Original_Index,Market,Symbol,Price
1,1,Spot,0GUSDT,0.188700
2,3,Spot,1000CATUSDT,0.001964
3,5,Spot,1000CHEEMSUSDT,0.000551
4,7,Spot,1000SATSUSDT,0.000011
5,8,Spot,1INCHDOWNUSDT,0.000000
...,...,...,...,...
735,1066,Spot,ZKPUSDT,0.046800
736,1068,Spot,ZKUSDT,0.009580
737,1070,Spot,ZROUSDT,1.111000
738,1071,Spot,ZRXUSDT,0.104000


In [22]:
usdc_only

,Original_Index,Market,Symbol,Price
1,0,Spot,0GUSDC,0.188500
2,2,Spot,1000CATUSDC,0.001969
3,4,Spot,1000CHEEMSUSDC,0.000551
4,6,Spot,1000SATSUSDC,0.000011
5,10,Spot,1INCHUSDC,0.092500
...,...,...,...,...
331,1063,Spot,ZKCUSDC,0.048300
332,1065,Spot,ZKPUSDC,0.046700
333,1067,Spot,ZKUSDC,0.009580
334,1069,Spot,ZROUSDC,1.110000


In [27]:
import pandas as pd

# 1. Filter out 0-price/delisted assets and ensure pairs end in USDT or USDC
clean_df = usdt_usdc_df[
    (usdt_usdc_df['Price'] > 0) & 
    (usdt_usdc_df['Symbol'].str.endswith(('USDT', 'USDC')))
].copy()

# 2. Extract Base coin and Quote currency
clean_df['Quote'] = clean_df['Symbol'].apply(lambda x: 'USDT' if x.endswith('USDT') else 'USDC')
clean_df['Base'] = clean_df['Symbol'].apply(lambda x: x[:-4])

# 3. Separate USDT and USDC pairs
usdt_df = clean_df[clean_df['Quote'] == 'USDT'][['Market', 'Base', 'Price']].rename(columns={'Price': 'USDT_Price'})
usdc_df = clean_df[clean_df['Quote'] == 'USDC'][['Market', 'Base', 'Price']].rename(columns={'Price': 'USDC_Price'})

# 4. Merge to isolate coins trading against BOTH USDT and USDC
dual_df = pd.merge(usdt_df, usdc_df, on=['Market', 'Base'], how='inner')

# 5. Calculate absolute difference and percentage spread
dual_df['Price_Difference'] = dual_df['USDT_Price'] - dual_df['USDC_Price']
dual_df['Spread_%'] = (dual_df['Price_Difference'] / dual_df['USDC_Price']) * 100

# 6. Clean up column names and sort by highest percentage difference
dual_df.rename(columns={'Base': 'Symbol'}, inplace=True)
dual_df['Abs_Spread'] = dual_df['Spread_%'].abs()
dual_df.sort_values(by='Abs_Spread', ascending=False, inplace=True)
dual_df.drop(columns=['Abs_Spread'], inplace=True)

print(f"Total coins trading on both USDT & USDC: {len(dual_df)}")
display(dual_df)

Total coins trading on both USDT & USDC: 317


,Original_Index,Market,Symbol,USDT_Price,USDC_Price,Price_Difference,Spread_%
1,295,Spot,WIN,0.00003379,0.0000973,-0.00006351,-65.27235355
2,58,Spot,BTTC,0.00000031,0.00000078,-0.00000047,-60.25641026
3,214,Spot,PUNDIX,0.0946,0.158,-0.0634,-40.12658228
4,144,Spot,JUV,0.316,0.514,-0.198,-38.52140078
5,257,Spot,SUSHI,0.1977,0.1539,0.0438,28.46003899
...,...,...,...,...,...,...,...
313,245,Spot,SOL,104.07,104.07,0,0
314,239,Spot,SHIB,0.00000534,0.00000534,0,0
315,237,Spot,SENT,0.0133,0.0133,0,0
316,236,Spot,SEI,0.04867,0.04867,0,0


In [33]:
import aiohttp
import asyncio
import time
import pandas as pd
import numpy as np

# Disable scientific notation globally
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')

# Function 1: Async Fetch from Binance
async def get_spot_and_coin_tickers():
    endpoints = {
        "Spot": "https://api.binance.com/api/v3/ticker/price",
        }
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [session.get(url) for url in endpoints.values()]
        responses = await asyncio.gather(*tasks)
        results = []
        for market, response in zip(endpoints.keys(), responses):
            if response.status == 200:
                data = await response.json()
                for item in data:
                    results.append({
                        "Market": market,
                        "Symbol": item.get("symbol"),
                        "Price": float(item.get("price")),
                    })
    return pd.DataFrame(results)

# Function 2: Filter Active USDT & USDC Pairs
def filter_usdt_usdc(df):
    return df[
        (df['Price'] > 0) & 
        (df['Symbol'].str.endswith(('USDT', 'USDC')))
    ].copy()

# Function 3: Run Match & Spread Analysis
def run_dual_df(df):
    clean_df = df.copy()
    clean_df['Quote'] = clean_df['Symbol'].apply(lambda x: 'USDT' if x.endswith('USDT') else 'USDC')
    clean_df['Base'] = clean_df['Symbol'].apply(lambda x: x[:-4])

    usdt_df = clean_df[clean_df['Quote'] == 'USDT'][['Market', 'Base', 'Price']].rename(columns={'Price': 'USDT_Price'})
    usdc_df = clean_df[clean_df['Quote'] == 'USDC'][['Market', 'Base', 'Price']].rename(columns={'Price': 'USDC_Price'})

    dual_df = pd.merge(usdt_df, usdc_df, on=['Market', 'Base'], how='inner')
    dual_df['Price_Difference'] = dual_df['USDT_Price'] - dual_df['USDC_Price']
    dual_df['Spread_%'] = (dual_df['Price_Difference'] / dual_df['USDC_Price']) * 100

    dual_df.rename(columns={'Base': 'Symbol'}, inplace=True)
    dual_df['Abs_Spread'] = dual_df['Spread_%'].abs()
    dual_df.sort_values(by='Abs_Spread', ascending=False, inplace=True)
    dual_df.drop(columns=['Abs_Spread'], inplace=True)
    
    return dual_df

# Execution Pipeline
tickers = await get_spot_and_coin_tickers()
usdt_usdc_df = filter_usdt_usdc(tickers)
dual_df = run_dual_df(usdt_usdc_df)

display(dual_df)

,Original_Index,Market,Symbol,USDT_Price,USDC_Price,Price_Difference,Spread_%
1,26,Spot,WIN,0.00003343,0.0000973,-0.00006387,-65.64234327
2,107,Spot,BTTC,0.0000003,0.00000078,-0.00000048,-61.53846154
3,76,Spot,PUNDIX,0.0936,0.158,-0.0644,-40.75949367
4,69,Spot,JUV,0.313,0.514,-0.201,-39.10505837
5,54,Spot,SUSHI,0.1927,0.1539,0.0388,25.21117609
...,...,...,...,...,...,...,...
313,51,Spot,RSR,0.00136,0.00136,0,0
314,49,Spot,DOT,0.859,0.859,0,0
315,130,Spot,FDUSD,0.9994,0.9994,0,0
316,36,Spot,EUR,1.1597,1.1597,0,0


In [34]:
from datetime import datetime

# Cell 1: Master execution cell
print(f"Fetching fresh Binance data at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

tickers = await get_spot_and_coin_tickers()
usdt_usdc_df = filter_usdt_usdc(tickers)
dual_df = run_dual_df(usdt_usdc_df)

display(dual_df)

Fetching fresh Binance data at: 2026-09-04 18:13:22


,Original_Index,Market,Symbol,USDT_Price,USDC_Price,Price_Difference,Spread_%
1,26,Spot,WIN,0.00003346,0.0000973,-0.00006384,-65.61151079
2,107,Spot,BTTC,0.0000003,0.00000078,-0.00000048,-61.53846154
3,76,Spot,PUNDIX,0.0932,0.158,-0.0648,-41.01265823
4,69,Spot,JUV,0.313,0.514,-0.201,-39.10505837
5,54,Spot,SUSHI,0.1925,0.1539,0.0386,25.08122157
...,...,...,...,...,...,...,...
313,104,Spot,ACH,0.00526,0.00526,0,0
314,110,Spot,APE,0.139,0.139,0,0
315,113,Spot,OP,0.0971,0.0971,0,0
316,243,Spot,HOME,0.00599,0.00599,0,0


In [4]:
import aiohttp
import asyncio
import pandas as pd

# Global float formatting (no scientific notation)
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')

async def get_active_spot_arbitrage():
    # ThreadedResolver prevents Android DNS crashes in Pydroid 3
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())
    
    async with aiohttp.ClientSession(connector=connector) as session:
        info_url = "https://api.binance.com/api/v3/exchangeInfo"
        price_url = "https://api.binance.com/api/v3/ticker/price"
        
        # Fetch metadata and live prices simultaneously
        info_resp, price_resp = await asyncio.gather(
            session.get(info_url),
            session.get(price_url)
        )
        
        info_data = await info_resp.json()
        price_data = await price_resp.json()
        
        # Map active prices
        price_map = {item['symbol']: float(item['price']) for item in price_data if float(item['price']) > 0}
        
        # Filter strictly by status == 'TRADING'
        active_list = []
        for s in info_data.get('symbols', []):
            symbol = s.get('symbol')
            status = s.get('status')
            quote = s.get('quoteAsset')
            base = s.get('baseAsset')
            is_spot = s.get('isSpotTradingAllowed', True)
            
            # Filter out delisted, halted, or non-spot pairs
            if status == 'TRADING' and is_spot and quote in ['USDT', 'USDC']:
                if symbol in price_map:
                    active_list.append({
                        'Symbol': base,
                        'Quote': quote,
                        'Price': price_map[symbol]
                    })
                    
        df = pd.DataFrame(active_list)
        
        # Split USDT and USDC
        usdt_df = df[df['Quote'] == 'USDT'][['Symbol', 'Price']].rename(columns={'Price': 'USDT_Price'})
        usdc_df = df[df['Quote'] == 'USDC'][['Symbol', 'Price']].rename(columns={'Price': 'USDC_Price'})
        
        # Merge pairs active in BOTH markets
        dual_df = pd.merge(usdt_df, usdc_df, on='Symbol', how='inner')
        
        # Calculations
        dual_df['Price_Difference'] = dual_df['USDT_Price'] - dual_df['USDC_Price']
        dual_df['Spread_%'] = (dual_df['Price_Difference'] / dual_df['USDC_Price']) * 100
        
        # Sort by absolute highest percentage spread
        dual_df['Abs_Spread'] = dual_df['Spread_%'].abs()
        dual_df.sort_values(by='Abs_Spread', ascending=False, inplace=True)
        dual_df.drop(columns=['Abs_Spread'], inplace=True)
        
        return dual_df

# Execute call
dual_df = await get_active_spot_arbitrage()
print(f"Total Active Coins Trading on BOTH USDT & USDC: {len(dual_df)}")
display(dual_df)

Total Active Coins Trading on BOTH USDT & USDC: 268


,Original_Index,Symbol,USDT_Price,USDC_Price,Price_Difference,Spread_%
1,181,WCT,0.03882,0.03952,-0.0007,-1.77125506
2,199,LA,0.0675,0.0665,0.001,1.5037594
3,207,DOLO,0.02718,0.02691,0.00027,1.00334448
4,99,BEAMX,0.001551,0.001565,-0.000014,-0.89456869
5,110,PIXEL,0.00495,0.00499,-0.00004,-0.80160321
...,...,...,...,...,...,...
264,28,COTI,0.01414,0.01414,0,0
265,112,WIF,0.1997,0.1997,0,0
266,235,TURTLE,0.041,0.041,0,0
267,237,KITE,0.1342,0.1342,0,0


# A

In [26]:
# Latest version 
import aiohttp
import asyncio
import pandas as pd

# Global float formatting (no scientific notation)
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')

# Global in-memory cache for trading pair metadata
_SYMBOL_CACHE = {}

async def get_active_spot_arbitrage():
    global _SYMBOL_CACHE
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())
    
    async with aiohttp.ClientSession(connector=connector) as session:
        # Step 1: Lazy-load symbol metadata ONCE and cache it
        if not _SYMBOL_CACHE:
            info_url = "https://api.binance.com/api/v3/exchangeInfo?permissions=SPOT"
            async with session.get(info_url) as resp:
                info_data = await resp.json()
            
            # Map base assets that have active USDT & USDC trading pairs
            temp_map = {}
            for s in info_data.get('symbols', []):
                if s.get('status') == 'TRADING' and s.get('isSpotTradingAllowed', True):
                    quote = s.get('quoteAsset')
                    base = s.get('baseAsset')
                    if quote in ('USDT', 'USDC'):
                        temp_map.setdefault(base, {})[quote] = s.get('symbol')
            
            # Cache only base assets actively listed on BOTH quotes
            _SYMBOL_CACHE = {
                base: pairs for base, pairs in temp_map.items()
                if 'USDT' in pairs and 'USDC' in pairs
            }

        # Step 2: Fetch prices ONLY (~100ms ultra-fast call)
        price_url = "https://api.binance.com/api/v3/ticker/price"
        async with session.get(price_url) as resp:
            price_data = await resp.json()

        # Step 3: Fast hash-map dictionary lookup
        price_map = {
            item['symbol']: float(item['price']) 
            for item in price_data 
            if float(item['price']) > 0
        }

        # Step 4: Compute spreads in pure Python (instant computation)
        results = []
        for base, pairs in _SYMBOL_CACHE.items():
            usdt_sym = pairs['USDT']
            usdc_sym = pairs['USDC']
            
            if usdt_sym in price_map and usdc_sym in price_map:
                u_price = price_map[usdt_sym]
                c_price = price_map[usdc_sym]
                diff = u_price - c_price
                spread = (diff / c_price) * 100
                
                results.append({
                    'Symbol': base,
                    'USDT_Price': u_price,
                    'USDC_Price': c_price,
                    'Price_Difference': diff,
                    'Spread_%': spread,
                    'Abs_Spread': abs(spread)
                })

        # Step 5: Construct final DataFrame & sort
        dual_df = pd.DataFrame(results)
        dual_df.sort_values(by='Abs_Spread', ascending=False, inplace=True)
        dual_df.drop(columns=['Abs_Spread'], inplace=True)
        dual_df.reset_index(drop=True, inplace=True)
        
        return dual_df

# Execute call
dual_df = await get_active_spot_arbitrage()
print(f"Total Active Coins Trading on BOTH USDT & USDC: {len(dual_df)}")
display(dual_df.head(20))

Total Active Coins Trading on BOTH USDT & USDC: 268


,Original_Index,Symbol,USDT_Price,USDC_Price,Price_Difference,Spread_%
1,0,AIXBT,0.02206,0.02256,-0.0005,-2.21631206
2,1,LA,0.0689,0.0697,-0.0008,-1.14777618
3,2,ACE,0.1766,0.1784,-0.0018,-1.00896861
4,3,MET,0.192,0.1905,0.0015,0.78740157
5,4,DYM,0.01539,0.01527,0.00012,0.78585462
6,5,KERNEL,0.0407,0.0404,0.0003,0.74257426
7,6,TURTLE,0.0425,0.0422,0.0003,0.71090047
8,7,MITO,0.018,0.01788,0.00012,0.67114094
9,8,LUNA,0.0472,0.0469,0.0003,0.63965885
10,9,LISTA,0.0846,0.0841,0.0005,0.59453032


In [3]:
# latest version with real-time update
import aiohttp
import asyncio
import pandas as pd
from datetime import datetime
from IPython.display import display, clear_output

# Global float formatting
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')

_SYMBOL_CACHE = {}

async def init_active_symbol_cache(session):
    """Fetches exchange metadata ONCE to filter strictly active TRADING spot pairs."""
    global _SYMBOL_CACHE
    if not _SYMBOL_CACHE:
        info_url = "https://api.binance.com/api/v3/exchangeInfo?permissions=SPOT"
        async with session.get(info_url) as resp:
            if resp.status != 200:
                raise RuntimeError(f"Failed to fetch exchange info: HTTP {resp.status}")
            info_data = await resp.json()
        
        temp_map = {}
        for s in info_data.get('symbols', []):
            if s.get('status') == 'TRADING' and s.get('isSpotTradingAllowed', True):
                quote = s.get('quoteAsset')
                base = s.get('baseAsset')
                if quote in ('USDT', 'USDC'):
                    temp_map.setdefault(base, {})[quote] = s.get('symbol')
        
        _SYMBOL_CACHE = {
            base: pairs for base, pairs in temp_map.items()
            if 'USDT' in pairs and 'USDC' in pairs
        }
    return _SYMBOL_CACHE

async def run_clean_realtime_arbitrage(poll_interval=0.2, min_spread_percent=0.0, top_n=15):
    timeout = aiohttp.ClientTimeout(total=5)
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        print("Initializing active symbol cache...")
        cache = await init_active_symbol_cache(session)
        
        price_url = "https://api.binance.com/api/v3/ticker/price"
        
        while True:
            try:
                timestamp = datetime.now()
                
                async with session.get(price_url) as resp:
                    if resp.status != 200:
                        await asyncio.sleep(1)
                        continue
                    price_data = await resp.json()
                
                price_map = {
                    item['symbol']: float(item['price']) 
                    for item in price_data 
                    if float(item['price']) > 0
                }
                
                results = []
                for base, pairs in cache.items():
                    u_sym, c_sym = pairs['USDT'], pairs['USDC']
                    
                    if u_sym in price_map and c_sym in price_map:
                        u_price = price_map[u_sym]
                        c_price = price_map[c_sym]
                        
                        if u_price <= 0 or c_price <= 0:
                            continue
                            
                        diff = u_price - c_price
                        spread = (diff / c_price) * 100
                        
                        if abs(spread) >= min_spread_percent:
                            results.append({
                                'Symbol': base,
                                'USDT_Price': u_price,
                                'USDC_Price': c_price,
                                'Spread_%': spread,
                                'Abs_Spread': abs(spread)
                            })
                
                if results:
                    df = pd.DataFrame(results)
                    df.sort_values(by='Abs_Spread', ascending=False, inplace=True)
                    
                    total_pairs = len(df)
                    df = df[['Symbol', 'USDT_Price', 'USDC_Price', 'Spread_%']]
                    
                    if top_n:
                        df = df.head(top_n)
                    
                    # Set Symbol as the row index to eliminate the extra 0, 1, 2... number column
                    df.set_index('Symbol', inplace=True)
                    
                    clear_output(wait=True)
                    print(f"⚡ REAL-TIME ARBITRAGE SCANNER [{timestamp.strftime('%H:%M:%S.%f')[:-3]}] | Active Dual Pairs: {total_pairs}")
                    display(df)
                else:
                    clear_output(wait=True)
                    print(f"[{timestamp.strftime('%H:%M:%S')}] Fetching prices... No pairs matched criteria.")
                
                await asyncio.sleep(poll_interval)
                
            except asyncio.CancelledError:
                print("\n[Stopped] Execution halted.")
                break
            except Exception as e:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Loop Error: {e}")
                await asyncio.sleep(1)

# Run scanner
try:
    await run_clean_realtime_arbitrage(poll_interval=1,min_spread_percent=0.6, top_n=10)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[Stopped] Live arbitrage stream halted.")

⚡ REAL-TIME ARBITRAGE SCANNER [17:24:12.365] | Active Dual Pairs: 8


,Original_Index,USDT_Price,USDC_Price,Spread_%
1,AIXBT,0.02209,0.02256,-2.08333333
2,LA,0.0692,0.0684,1.16959064
3,COW,0.1276,0.1287,-0.85470085
4,BARD,0.1237,0.1246,-0.7223114
5,SOPH,0.00439,0.00436,0.68807339
6,GUN,0.00312,0.00314,-0.63694268
7,SOLV,0.00322,0.00324,-0.61728395
8,W,0.00998,0.00992,0.60483871



[Stopped] Execution halted.


# B

In [7]:
import aiohttp
import asyncio
import pandas as pd
from datetime import datetime
from IPython.display import display, clear_output
import hashlib
import hmac
import json
import time
from urllib.parse import urlencode
import websockets
import math

# =====================================================================
# CONFIGURATION
# =====================================================================

TEST_MODE = False # CHANGE TO 'False' FOR LIVE TRADING
TRADE_QUOTE_AMOUNT = 15.0  # Amount in USDT/USDC to spend on the first leg
TARGET_SPREAD_PERCENT = 0.7  # Minimum spread threshold to trigger execution

WS_URL = "wss://ws-api.binance.com:443/ws-api/v3"

# Global Formatting & Cache
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')
_SYMBOL_CACHE = {}
ws_client = None


# =====================================================================
# WEBSOCKET & EXECUTION UTILITIES
# =====================================================================
async def get_ws_connection():
    global ws_client
    is_closed = True
    if ws_client is not None:
        if hasattr(ws_client, "close_code"):
            is_closed = ws_client.close_code is not None
        elif hasattr(ws_client, "closed"):
            is_closed = ws_client.closed

    if ws_client is None or is_closed:
        ws_client = await websockets.connect(WS_URL)
    return ws_client

def generate_signature(params: dict, secret: str) -> str:
    sorted_params = dict(sorted(params.items()))
    query_string = urlencode(sorted_params)
    return hmac.new(secret.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256).hexdigest()

async def send_order(symbol_formatted: str, side: str, params_extra: dict, test_mode: bool = False):
    ws = await get_ws_connection()
    now = int(time.time() * 1000)

    params = {
        "apiKey": API_KEY,
        "symbol": symbol_formatted,
        "side": side.upper(),
        "type": "MARKET",
        "newOrderRespType": "FULL",  # Returns exact fills and cumulative totals instantly
        "timestamp": now,
        **params_extra,
    }
    params["signature"] = generate_signature(params, API_SECRET)

    method = "order.test" if test_mode else "order.place"
    payload = {
        "id": f"ord_{now}",
        "method": method,
        "params": params,
    }

    t0 = time.perf_counter()
    await ws.send(json.dumps(payload))
    res = json.loads(await ws.recv())
    latency_ms = (time.perf_counter() - t0) * 1000

    return res, latency_ms

def round_step_size(quantity: float, step_size: float) -> str:
    if step_size <= 0:
        return str(quantity)
    precision = int(round(-math.log10(step_size)))
    if precision <= 0:
        factor = 10 ** abs(precision)
        truncated = math.floor(quantity / factor) * factor
        return f"{int(truncated)}"
    factor = 10**precision
    truncated = math.floor(quantity * factor) / factor
    return f"{truncated:.{precision}f}"

def get_average_fill_price(order_res: dict, fallback_price: float = 0.0) -> float:
    """Calculates true average fill price from Binance FULL response, with a live ticker fallback."""
    result_data = order_res.get("result", {})
    executed_qty = float(result_data.get("executedQty", 0.0))
    cumm_quote = float(result_data.get("cummulativeQuoteQty", 0.0))
    
    if executed_qty > 0 and cumm_quote > 0:
        return cumm_quote / executed_qty
        
    # Fallback to individual fills list if present
    fills = result_data.get("fills", [])
    if fills:
        total_cost = sum(float(f["price"]) * float(f["qty"]) for f in fills)
        total_qty = sum(float(f["qty"]) for f in fills)
        if total_qty > 0:
            return total_cost / total_qty
            
    # Ultimate fallback to live market price if exchange payload omits cumulative totals
    return fallback_price


# =====================================================================
# REAL-TIME SCANNER & EXECUTION ENGINE
# =====================================================================
async def init_active_symbol_cache(session):
    global _SYMBOL_CACHE
    if not _SYMBOL_CACHE:
        info_url = "https://api.binance.com/api/v3/exchangeInfo?permissions=SPOT"
        async with session.get(info_url) as resp:
            info_data = await resp.json()
        
        temp_map = {}
        for s in info_data.get('symbols', []):
            if s.get('status') == 'TRADING' and s.get('isSpotTradingAllowed', True):
                quote, base = s.get('quoteAsset'), s.get('baseAsset')
                if quote in ('USDT', 'USDC'):
                    step_size = 1.0
                    for f in s.get('filters', []):
                        if f.get('filterType') == 'LOT_SIZE':
                            step_size = float(f.get('stepSize', '1.0'))
                            break
                    
                    temp_map.setdefault(base, {})[quote] = {
                        'symbol': s.get('symbol'),
                        'step_size': step_size
                    }
        
        _SYMBOL_CACHE = {
            base: pairs for base, pairs in temp_map.items()
            if 'USDT' in pairs and 'USDC' in pairs
        }
    return _SYMBOL_CACHE

async def run_arbitrage_bot(poll_interval=0.2, top_n=10):
    timeout = aiohttp.ClientTimeout(total=5)
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())
    trade_executed = False  
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        print("Initializing active symbol cache & fetching lot sizes...")
        cache = await init_active_symbol_cache(session)
        price_url = "https://api.binance.com/api/v3/ticker/price"
        
        while not trade_executed:
            try:
                timestamp = datetime.now()
                async with session.get(price_url) as resp:
                    if resp.status != 200:
                        await asyncio.sleep(1)
                        continue
                    price_data = await resp.json()
                
                price_map = {item['symbol']: float(item['price']) for item in price_data if float(item['price']) > 0}
                results = []
                
                for base, pairs in cache.items():
                    u_sym_data, c_sym_data = pairs['USDT'], pairs['USDC']
                    u_sym, c_sym = u_sym_data['symbol'], c_sym_data['symbol']
                    
                    if u_sym in price_map and c_sym in price_map:
                        u_price, c_price = price_map[u_sym], price_map[c_sym]
                        
                        # Evaluate both spread directions
                        spread_usdt_expensive = ((u_price - c_price) / c_price) * 100  # Buy USDC, Sell USDT
                        spread_usdt_cheaper = ((c_price - u_price) / u_price) * 100    # Buy USDT, Sell USDC
                        
                        target_direction = None
                        active_spread = 0.0
                        
                        if spread_usdt_expensive >= TARGET_SPREAD_PERCENT:
                            target_direction = "USDT_EXPENSIVE"
                            active_spread = spread_usdt_expensive
                        elif spread_usdt_cheaper >= TARGET_SPREAD_PERCENT:
                            target_direction = "USDT_CHEAPER"
                            active_spread = spread_usdt_cheaper

                        # Log max spread for display table
                        max_spread = spread_usdt_expensive if abs(spread_usdt_expensive) >= abs(spread_usdt_cheaper) else -spread_usdt_cheaper
                        if abs(max_spread) >= 0.1:
                            results.append({
                                'Symbol': base,
                                'USDT_Price': u_price,
                                'USDC_Price': c_price,
                                'Spread_%': max_spread,
                                'Abs_Spread': abs(max_spread)
                            })
                        
                        # EXECUTION TRIGGER
                        if target_direction and not trade_executed:
                            clear_output(wait=True)
                            print("="*75)
                            print(f"🚀 ARBITRAGE TRIGGERED! Asset: {base} | Mode: {target_direction}")
                            print(f"Estimated Spread: {active_spread:.3f}% (USDT: {u_price} | USDC: {c_price})")
                            print("="*75)
                            
                            trade_executed = True 
                            
                            # Set leg properties dynamically based on direction
                            if target_direction == "USDT_EXPENSIVE":
                                buy_symbol = c_sym
                                sell_symbol = u_sym
                                sell_step_size = u_sym_data['step_size']
                                fallback_buy_price = c_price
                                fallback_sell_price = u_price
                            else:
                                buy_symbol = u_sym
                                sell_symbol = c_sym
                                sell_step_size = c_sym_data['step_size']
                                fallback_buy_price = u_price
                                fallback_sell_price = c_price
                            
                            # --- LEG 1: BUY CHEAPER PAIR ---
                            buy_res, buy_latency = await send_order(
                                symbol_formatted=buy_symbol,
                                side="BUY",
                                params_extra={"quoteOrderQty": str(TRADE_QUOTE_AMOUNT)},
                                test_mode=TEST_MODE
                            )
                            
                            mode = "TEST" if TEST_MODE else "LIVE"
                            executed_qty = 0.0
                            buy_exec_price = 0.0
                            result_data = {}
                            
                            if TEST_MODE:
                                print(f"[{mode} BUY] Pair: {buy_symbol} | RTT: {buy_latency:.2f} ms")
                                executed_qty = TRADE_QUOTE_AMOUNT / fallback_buy_price
                                buy_exec_price = fallback_buy_price
                            else:
                                if "error" in buy_res or buy_res.get("status") != 200:
                                    print(f"❌ BUY Error Response: {buy_res.get('error', buy_res)}")
                                    break
                                    
                                result_data = buy_res.get("result", {})
                                status = result_data.get("status")
                                
                                if status in ["FILLED", "PARTIALLY_FILLED"]:
                                    executed_qty = float(result_data.get("executedQty", 0.0))
                                    buy_exec_price = get_average_fill_price(buy_res, fallback_price=fallback_buy_price)
                                    print(f"[{mode} BUY] Pair: {buy_symbol} | Real Fill Price: {buy_exec_price:.8f} | Gross Qty: {executed_qty} | RTT: {buy_latency:.2f} ms")
                                else:
                                    print(f"❌ BUY failed. Status: {status}. Response: {buy_res}")
                                    break
                            
                            # --- LEG 2: SELL EXPENSIVE PAIR ---
                            if executed_qty > 0:
                                actual_qty_to_sell = executed_qty
                                if not TEST_MODE:
                                    # Subtract commissions paid in base asset
                                    for f in result_data.get("fills", []):
                                        if f.get("commissionAsset") == base:
                                            actual_qty_to_sell -= float(f.get("commission", 0.0))
                                
                                sell_qty_str = round_step_size(actual_qty_to_sell, sell_step_size)
                                
                                sell_res, sell_latency = await send_order(
                                    symbol_formatted=sell_symbol,
                                    side="SELL",
                                    params_extra={"quantity": sell_qty_str},
                                    test_mode=TEST_MODE
                                )
                                
                                if TEST_MODE:
                                    sell_exec_price = fallback_sell_price
                                    print(f"[{mode} SELL] Pair: {sell_symbol} | Qty: {sell_qty_str} | RTT: {sell_latency:.2f} ms")
                                else:
                                    if "error" in sell_res or sell_res.get("status") != 200:
                                        print(f"❌ CRITICAL SELL ERROR: Binance rejected the order. Details: {sell_res}")
                                        break
                                        
                                    sell_result_data = sell_res.get("result", {})
                                    sell_order_status = sell_result_data.get("status")
                                    
                                    if sell_order_status not in ["FILLED", "PARTIALLY_FILLED"]:
                                        print(f"❌ SELL order failed to fill. Status: {sell_order_status}. Payload: {sell_res}")
                                        break

                                    sell_exec_price = get_average_fill_price(sell_res, fallback_price=fallback_sell_price)
                                    print(f"[{mode} SELL] Pair: {sell_symbol} | Real Fill Price: {sell_exec_price:.8f} | Net Qty: {sell_qty_str} | RTT: {sell_latency:.2f} ms")
                                
                                # --- CALCULATE REALIZED SPREAD ---
                                if buy_exec_price > 0 and sell_exec_price > 0:
                                    realized_spread = ((sell_exec_price - buy_exec_price) / buy_exec_price) * 100
                                    print("-" * 75)
                                    print(f"📊 EXECUTION SUMMARY REPORT:")
                                    print(f"   • Real Buy Price:  {buy_exec_price:.8f} ({buy_symbol})")
                                    print(f"   • Real Sell Price: {sell_exec_price:.8f} ({sell_symbol})")
                                    print(f"   • Realized Spread: {realized_spread:+.3f}%")
                                    print("-" * 75)
                            
                            print("\n✅ Arbitrage execution complete. Shutting down scanner.")
                            break 
                
                if not trade_executed and results:
                    df = pd.DataFrame(results).sort_values(by='Abs_Spread', ascending=False)
                    df = df[['Symbol', 'USDT_Price', 'USDC_Price', 'Spread_%']].head(top_n)
                    df.set_index('Symbol', inplace=True)
                    
                    clear_output(wait=True)
                    print(f"⚡ SCANNER [{timestamp.strftime('%H:%M:%S.%f')[:-3]}] | Target Spread: >{TARGET_SPREAD_PERCENT}% | Status: Watching...")
                    display(df)
                
                if trade_executed:
                    break
                    
                await asyncio.sleep(poll_interval)
                
            except asyncio.CancelledError:
                break
            except Exception as e:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Loop Error: {e}")
                await asyncio.sleep(1)

# =====================================================================
# START SCRIPT
# =====================================================================
try:
    await run_arbitrage_bot(poll_interval=1.0, top_n=10)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[Stopped] Live arbitrage stream halted.")

🚀 ARBITRAGE TRIGGERED! Asset: AIXBT | Mode: USDT_CHEAPER
Estimated Spread: 2.452% (USDT: 0.02202 | USDC: 0.02256)
[LIVE BUY] Pair: AIXBTUSDT | Real Fill Price: 0.02204000 | Gross Qty: 680.5 | RTT: 235.75 ms
[LIVE SELL] Pair: AIXBTUSDC | Real Fill Price: 0.02199000 | Net Qty: 679.8 | RTT: 229.35 ms
---------------------------------------------------------------------------
📊 EXECUTION SUMMARY REPORT:
   • Real Buy Price:  0.02204000 (AIXBTUSDT)
   • Real Sell Price: 0.02199000 (AIXBTUSDC)
   • Realized Spread: -0.227%
---------------------------------------------------------------------------

✅ Arbitrage execution complete. Shutting down scanner.


In [4]:
# APIs
API_KEY = "HwpbVi9ineNbjmNw5MqHj6N7GNG2IUclygYwo7x7gkLyDievhYq7tTGYKo9sBXCT"
API_SECRET = "hrnzMAnag1rRUlZzaXRJN2jIAKpUnqxVISfc6BOZg6ZNMmjLcaw3ty6Rcuv8FZu6"


In [9]:
# My IP
import urllib.request

public_ip = (
    urllib.request.urlopen("https://api.ipify.org").read().decode("utf8")
)
print(f"Your current Public IP for Binance Whitelisting: {public_ip}")

Your current Public IP for Binance Whitelisting: 45.121.88.36


In [5]:
# Binance check with total acc val
import asyncio
import hashlib
import hmac
import json
import time
from urllib.parse import urlencode
import websockets

WS_URL = "wss://ws-api.binance.com:443/ws-api/v3"

def generate_signature(params: dict, secret: str) -> str:
    sorted_params = dict(sorted(params.items()))
    query_string = urlencode(sorted_params)
    return hmac.new(
        secret.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256
    ).hexdigest()

async def get_total_spot_value():
    async with websockets.connect(WS_URL) as ws:
        now = int(time.time() * 1000)

        # 1. Fetch account balances
        bal_params = {
            "apiKey": API_KEY,
            "omitZeroBalances": "true",
            "timestamp": now,
        }
        bal_params["signature"] = generate_signature(bal_params, API_SECRET)

        await ws.send(json.dumps({
            "id": f"bal_{now}",
            "method": "account.status",
            "params": bal_params
        }))
        bal_response = json.loads(await ws.recv())

        if bal_response.get("status") != 200:
            print(f"❌ Failed to fetch balances: {bal_response.get('error', bal_response)}")
            return

        balances = bal_response.get("result", {}).get("balances", [])
        non_zero = [b for b in balances if float(b["free"]) > 0 or float(b["locked"]) > 0]

        # 2. Fetch all ticker prices to value assets against USDT
        await ws.send(json.dumps({
            "id": f"ticker_{now}",
            "method": "ticker.price"
        }))
        ticker_response = json.loads(await ws.recv())
        
        prices = {}
        if ticker_response.get("status") == 200:
            for t in ticker_response.get("result", []):
                prices[t["symbol"]] = float(t["price"])

        # 3. Calculate total value in USDT
        total_usdt_value = 0.0
        print("=== 💰 SPOT PORTFOLIO VALUATION ===")

        for b in non_zero:
            asset = b["asset"]
            qty = float(b["free"]) + float(b["locked"])
            
            if asset in ["USDT", "USDC", "FDUSD"]:
                # Treat stablecoins roughly 1:1
                val_in_usdt = qty
                if asset != "USDT":
                    # Check if USDC/FDUSD has a direct USDT pair just in case
                    pair = f"{asset}USDT"
                    if pair in prices:
                        val_in_usdt = qty * prices[pair]
                print(f"• {asset}: {qty} (~{val_in_usdt:.2f} USDT)")
            else:
                pair = f"{asset}USDT"
                if pair in prices:
                    val_in_usdt = qty * prices[pair]
                    print(f"• {asset}: {qty} (~{val_in_usdt:.2f} USDT)")
                else:
                    # Try reverse or fallback if needed, else 0
                    val_in_usdt = 0.0
                    print(f"• {asset}: {qty} (No direct USDT price found)")

            total_usdt_value += val_in_usdt

        print("----------------------------------------")
        print(f"🌟 ESTIMATED TOTAL SPOT VALUE: ~{total_usdt_value:,.2f} USDT")

# Execute directly in Jupyter
await get_total_spot_value()

=== 💰 SPOT PORTFOLIO VALUATION ===
• USDT: 89.77907733 (~89.78 USDT)
• USDC: 9.6381635 (~9.64 USDT)
• DYM: 0.075935 (~0.00 USDT)
• CATI: 0.032005 (~0.00 USDT)
• TURBO: 0.11195 (~0.00 USDT)
• VELODROME: 0.01385 (~0.00 USDT)
• ANIME: 0.08622 (~0.00 USDT)
• MUBARAK: 0.057395 (~0.00 USDT)
----------------------------------------
🌟 ESTIMATED TOTAL SPOT VALUE: ~99.42 USDT


In [29]:
# Check binance balance 
import asyncio
import hashlib
import hmac
import json
import time
from urllib.parse import urlencode
import websockets

WS_URL = "wss://ws-api.binance.com:443/ws-api/v3"


def generate_signature(params: dict, secret: str) -> str:
    sorted_params = dict(sorted(params.items()))
    query_string = urlencode(sorted_params)
    return hmac.new(
        secret.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256
    ).hexdigest()


async def check_balances():
    async with websockets.connect(WS_URL) as ws:
        now = int(time.time() * 1000)

        # Using lowercase string "true" prevents Python boolean urlencode signature mismatch
        params = {
            "apiKey": API_KEY,
            "omitZeroBalances": "true",
            "timestamp": now,
        }
        params["signature"] = generate_signature(params, API_SECRET)

        payload = {
            "id": f"bal_check_{now}",
            "method": "account.status",
            "params": params,
        }

        await ws.send(json.dumps(payload))
        response = json.loads(await ws.recv())

        if response.get("status") != 200:
            print(f"❌ Failed to fetch balances: {response.get('error', response)}")
            return

        balances = response.get("result", {}).get("balances", [])

        # Extract specific USDT & USDC balances
        usdt = next(
            (b for b in balances if b["asset"] == "USDT"),
            {"free": "0.00", "locked": "0.00"},
        )
        usdc = next(
            (b for b in balances if b["asset"] == "USDC"),
            {"free": "0.00", "locked": "0.00"},
        )

        print("=== 💵 TARGET STABLECOIN BALANCES ===")
        print(f"USDT : {usdt['free']} Free | {usdt['locked']} Locked")
        print(f"USDC : {usdc['free']} Free | {usdc['locked']} Locked")

        print("\n=== 💼 ALL NON-ZERO ACCOUNT HOLDINGS ===")
        non_zero = [
            b for b in balances if float(b["free"]) > 0 or float(b["locked"]) > 0
        ]
        if not non_zero:
            print("No active non-zero assets found.")
        else:
            for b in non_zero:
                print(
                    f"• {b['asset'].ljust(6)}: {b['free']} (Free) | {b['locked']} (Locked)"
                )


# Execute directly in Jupyter
await check_balances()

=== 💵 TARGET STABLECOIN BALANCES ===
USDT : 89.77907733 Free | 0.00000000 Locked
USDC : 9.63816350 Free | 0.00000000 Locked

=== 💼 ALL NON-ZERO ACCOUNT HOLDINGS ===
• USDT  : 89.77907733 (Free) | 0.00000000 (Locked)
• USDC  : 9.63816350 (Free) | 0.00000000 (Locked)
• DYM   : 0.07593500 (Free) | 0.00000000 (Locked)
• CATI  : 0.03200500 (Free) | 0.00000000 (Locked)
• TURBO : 0.11195000 (Free) | 0.00000000 (Locked)
• VELODROME: 0.01385000 (Free) | 0.00000000 (Locked)
• ANIME : 0.08622000 (Free) | 0.00000000 (Locked)
• MUBARAK: 0.05739500 (Free) | 0.00000000 (Locked)


In [7]:
# 1
import aiohttp
import asyncio
import pandas as pd
from datetime import datetime
from IPython.display import display, clear_output
import hashlib
import hmac
import json
import time
from urllib.parse import urlencode
import websockets
import math

# =====================================================================
# CONFIGURATION
# =====================================================================

TEST_MODE = True  # CHANGE TO 'False' FOR LIVE TRADING
TRADE_QUOTE_AMOUNT = 101.0  # Amount in USDC to spend on the first leg
TARGET_SPREAD_PERCENT = 0.5  # Trigger buy if USDC is this much cheaper than USDT

WS_URL = "wss://ws-api.binance.com:443/ws-api/v3"

# Global Formatting & Cache
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')
_SYMBOL_CACHE = {}
ws_client = None


# =====================================================================
# WEBSOCKET & EXECUTION UTILITIES
# =====================================================================
async def get_ws_connection():
    global ws_client
    is_closed = True
    if ws_client is not None:
        if hasattr(ws_client, "close_code"):
            is_closed = ws_client.close_code is not None
        elif hasattr(ws_client, "closed"):
            is_closed = ws_client.closed

    if ws_client is None or is_closed:
        ws_client = await websockets.connect(WS_URL)
    return ws_client

def generate_signature(params: dict, secret: str) -> str:
    sorted_params = dict(sorted(params.items()))
    query_string = urlencode(sorted_params)
    return hmac.new(secret.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256).hexdigest()

def parse_symbol(raw_symbol: str):
    s = raw_symbol.upper().replace("/", "").replace("-", "").replace("_", "")
    quotes = ["USDT", "USDC", "FDUSD", "BUSD", "BTC", "ETH", "BNB", "EUR"]
    base = s
    for q in quotes:
        if s.endswith(q):
            base = s[: -len(q)]
            break
    return s, base

async def send_order(symbol_formatted: str, side: str, params_extra: dict, test_mode: bool = False):
    ws = await get_ws_connection()
    now = int(time.time() * 1000)

    params = {
        "apiKey": API_KEY,
        "symbol": symbol_formatted,
        "side": side.upper(),
        "type": "MARKET",
        "newOrderRespType": "ACK",
        "timestamp": now,
        **params_extra,
    }
    params["signature"] = generate_signature(params, API_SECRET)

    method = "order.test" if test_mode else "order.place"
    payload = {
        "id": f"ord_{now}",
        "method": method,
        "params": params,
    }

    t0 = time.perf_counter()
    await ws.send(json.dumps(payload))
    res = json.loads(await ws.recv())
    latency_ms = (time.perf_counter() - t0) * 1000

    return res, latency_ms

async def fast_buy(symbol: str, quote_amount: float = 101.0, test_mode: bool = False):
    symbol_formatted, _ = parse_symbol(symbol)
    res, latency_ms = await send_order(
        symbol_formatted=symbol_formatted,
        side="BUY",
        params_extra={"quoteOrderQty": str(quote_amount)},
        test_mode=test_mode,
    )
    mode_tag = "TEST" if test_mode else "LIVE"
    print(f"[{mode_tag} BUY] Pair: {symbol_formatted} | Spend: ${quote_amount} | RTT: {latency_ms:.2f} ms")
    return res

def round_step_size(quantity: float, step_size: float) -> str:
    if step_size <= 0:
        return str(quantity)
    precision = int(round(-math.log10(step_size)))
    if precision <= 0:
        factor = 10 ** abs(precision)
        truncated = math.floor(quantity / factor) * factor
        return f"{int(truncated)}"
    factor = 10**precision
    truncated = math.floor(quantity * factor) / factor
    return f"{truncated:.{precision}f}"

async def fast_sell_all(symbol: str, min_notional_usdt: float = 5.0, test_mode: bool = False):
    ws = await get_ws_connection()
    symbol_formatted, base_asset = parse_symbol(symbol)
    now = int(time.time() * 1000)

    # 1. Fetch Symbol Info
    info_payload = {"id": f"info_{now}", "method": "exchangeInfo", "params": {"symbol": symbol_formatted}}
    await ws.send(json.dumps(info_payload))
    info_res = json.loads(await ws.recv())

    step_size = 1.0
    filters = info_res.get("result", {}).get("symbols", [{}])[0].get("filters", [])
    for f in filters:
        if f.get("filterType") == "LOT_SIZE":
            step_size = float(f.get("stepSize", 1.0))
            break

    # 2. Fetch Balance
    bal_params = {"apiKey": API_KEY, "timestamp": now}
    bal_params["signature"] = generate_signature(bal_params, API_SECRET)
    await ws.send(json.dumps({"id": f"bal_{now}", "method": "account.status", "params": bal_params}))
    bal_res = json.loads(await ws.recv())

    free_bal = 0.0
    for b in bal_res.get("result", {}).get("balances", []):
        if b["asset"] == base_asset:
            free_bal = float(b["free"])
            break

    if free_bal <= 0:
        if test_mode:
            free_bal = 100.0  # Mock quantity for testing
        else:
            print(f"⚠️ Sell Skipped: No available balance for {base_asset}")
            return {"error": f"No balance for {base_asset}"}

    # 3. StepSize precision formatting
    qty_str = round_step_size(free_bal, step_size)
    if float(qty_str) <= 0:
        print(f"⚠️ Sell Skipped: Balance ({free_bal}) < stepSize ({step_size}).")
        return {"error": "Quantity below stepSize minimum"}

    # 4. Dispatch Order
    res, latency_ms = await send_order(
        symbol_formatted=symbol_formatted,
        side="SELL",
        params_extra={"quantity": qty_str},
        test_mode=test_mode,
    )
    mode_tag = "TEST" if test_mode else "LIVE"
    print(f"[{mode_tag} SELL] Symbol: {symbol_formatted} | Validated Qty: {qty_str} | RTT: {latency_ms:.2f} ms")
    return res


# =====================================================================
# REAL-TIME SCANNER & LOGIC ENGINE
# =====================================================================
async def init_active_symbol_cache(session):
    global _SYMBOL_CACHE
    if not _SYMBOL_CACHE:
        info_url = "https://api.binance.com/api/v3/exchangeInfo?permissions=SPOT"
        async with session.get(info_url) as resp:
            info_data = await resp.json()
        
        temp_map = {}
        for s in info_data.get('symbols', []):
            if s.get('status') == 'TRADING' and s.get('isSpotTradingAllowed', True):
                quote, base = s.get('quoteAsset'), s.get('baseAsset')
                if quote in ('USDT', 'USDC'):
                    temp_map.setdefault(base, {})[quote] = s.get('symbol')
        
        _SYMBOL_CACHE = {
            base: pairs for base, pairs in temp_map.items()
            if 'USDT' in pairs and 'USDC' in pairs
        }
    return _SYMBOL_CACHE

async def run_arbitrage_bot(poll_interval=0.2, top_n=10):
    timeout = aiohttp.ClientTimeout(total=5)
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())
    trade_executed = False  # Enforces the "one trade only" rule
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        print("Initializing active symbol cache...")
        cache = await init_active_symbol_cache(session)
        price_url = "https://api.binance.com/api/v3/ticker/price"
        
        while not trade_executed:
            try:
                timestamp = datetime.now()
                async with session.get(price_url) as resp:
                    if resp.status != 200:
                        await asyncio.sleep(1)
                        continue
                    price_data = await resp.json()
                
                price_map = {item['symbol']: float(item['price']) for item in price_data if float(item['price']) > 0}
                results = []
                
                for base, pairs in cache.items():
                    u_sym, c_sym = pairs['USDT'], pairs['USDC']
                    
                    if u_sym in price_map and c_sym in price_map:
                        u_price, c_price = price_map[u_sym], price_map[c_sym]
                        
                        # Spread positive means USDC is cheaper than USDT
                        spread = ((u_price - c_price) / c_price) * 100
                        
                        # EXECUTION TRIGGER (0.5% cheaper)
                        if spread >= TARGET_SPREAD_PERCENT and not trade_executed:
                            clear_output(wait=True)
                            print("="*60)
                            print(f"🚀 ARBITRAGE TRIGGERED! {base}")
                            print(f"USDT Price: {u_price} | USDC Price: {c_price} | Spread: {spread:.3f}%")
                            print("="*60)
                            
                            trade_executed = True # Locks execution so we only do this once
                            
                            # 1. Buy the cheaper pair (USDC)
                            await fast_buy(c_sym, quote_amount=TRADE_QUOTE_AMOUNT, test_mode=TEST_MODE)
                            
                            # Give Binance engine a tiny moment to update balance before selling
                            await asyncio.sleep(1.0)
                            
                            # 2. Sell on the expensive pair (USDT)
                            await fast_sell_all(u_sym, test_mode=TEST_MODE)
                            
                            print("\n✅ Arbitrage execution complete. Shutting down scanner.")
                            break # Exits the for-loop
                        
                        # For Display Purposes
                        if abs(spread) >= 0.1: # Only display spreads larger than 0.1% to save screen space
                            results.append({
                                'Symbol': base,
                                'USDT_Price': u_price,
                                'USDC_Price': c_price,
                                'Spread_%': spread,
                                'Abs_Spread': abs(spread)
                            })
                
                # Update visual display if we haven't executed a trade yet
                if not trade_executed and results:
                    df = pd.DataFrame(results).sort_values(by='Abs_Spread', ascending=False)
                    df = df[['Symbol', 'USDT_Price', 'USDC_Price', 'Spread_%']].head(top_n)
                    df.set_index('Symbol', inplace=True)
                    
                    clear_output(wait=True)
                    print(f"⚡ SCANNER [{timestamp.strftime('%H:%M:%S')}] | Target Spread: >{TARGET_SPREAD_PERCENT}% | Status: Watching...")
                    display(df)
                
                if trade_executed:
                    break # Exits the while-loop securely
                    
                await asyncio.sleep(poll_interval)
                
            except asyncio.CancelledError:
                break
            except Exception as e:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Loop Error: {e}")
                await asyncio.sleep(1)

# =====================================================================
# START SCRIPT
# =====================================================================
try:
    await run_arbitrage_bot(poll_interval=1.0, top_n=10)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[Stopped] Live arbitrage stream halted.")

🚀 ARBITRAGE TRIGGERED! COOKIE
USDT Price: 0.0118 | USDC Price: 0.0117 | Spread: 0.855%
[TEST BUY] Pair: COOKIEUSDC | Spend: $101.0 | RTT: 354.22 ms
[TEST SELL] Symbol: COOKIEUSDT | Validated Qty: 100.0 | RTT: 200.98 ms

✅ Arbitrage execution complete. Shutting down scanner.


In [9]:
# 2
import aiohttp
import asyncio
import pandas as pd
from datetime import datetime
from IPython.display import display, clear_output
import hashlib
import hmac
import json
import time
from urllib.parse import urlencode
import websockets
import math

# =====================================================================
# CONFIGURATION
# =====================================================================

TEST_MODE = True  # CHANGE TO 'False' FOR LIVE TRADING
TRADE_QUOTE_AMOUNT = 101.0  # Amount in USDC to spend on the first leg
TARGET_SPREAD_PERCENT = 0.5  # Trigger buy if USDC is >= 0.5% cheaper than USDT

WS_URL = "wss://ws-api.binance.com:443/ws-api/v3"

# Global Formatting & Cache
pd.set_option('display.float_format', lambda x: f'{x:.8f}'.rstrip('0').rstrip('.') if x != 0 else '0')
_SYMBOL_CACHE = {}
ws_client = None


# =====================================================================
# WEBSOCKET & EXECUTION UTILITIES
# =====================================================================
async def get_ws_connection():
    global ws_client
    is_closed = True
    if ws_client is not None:
        if hasattr(ws_client, "close_code"):
            is_closed = ws_client.close_code is not None
        elif hasattr(ws_client, "closed"):
            is_closed = ws_client.closed

    if ws_client is None or is_closed:
        ws_client = await websockets.connect(WS_URL)
    return ws_client

def generate_signature(params: dict, secret: str) -> str:
    sorted_params = dict(sorted(params.items()))
    query_string = urlencode(sorted_params)
    return hmac.new(secret.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256).hexdigest()

async def send_order(symbol_formatted: str, side: str, params_extra: dict, test_mode: bool = False):
    ws = await get_ws_connection()
    now = int(time.time() * 1000)

    params = {
        "apiKey": API_KEY,
        "symbol": symbol_formatted,
        "side": side.upper(),
        "type": "MARKET",
        "newOrderRespType": "FULL",  # CRITICAL FOR SPEED: Returns fill status & exact qty immediately
        "timestamp": now,
        **params_extra,
    }
    params["signature"] = generate_signature(params, API_SECRET)

    method = "order.test" if test_mode else "order.place"
    payload = {
        "id": f"ord_{now}",
        "method": method,
        "params": params,
    }

    t0 = time.perf_counter()
    await ws.send(json.dumps(payload))
    res = json.loads(await ws.recv())
    latency_ms = (time.perf_counter() - t0) * 1000

    return res, latency_ms

def round_step_size(quantity: float, step_size: float) -> str:
    if step_size <= 0:
        return str(quantity)
    precision = int(round(-math.log10(step_size)))
    if precision <= 0:
        factor = 10 ** abs(precision)
        truncated = math.floor(quantity / factor) * factor
        return f"{int(truncated)}"
    factor = 10**precision
    truncated = math.floor(quantity * factor) / factor
    return f"{truncated:.{precision}f}"


# =====================================================================
# REAL-TIME SCANNER & HYPER-SPEED LOGIC ENGINE
# =====================================================================
async def init_active_symbol_cache(session):
    """Fetches exchange metadata, caches pairs, and caches LOT_SIZE step sizes to eliminate execution lag."""
    global _SYMBOL_CACHE
    if not _SYMBOL_CACHE:
        info_url = "https://api.binance.com/api/v3/exchangeInfo?permissions=SPOT"
        async with session.get(info_url) as resp:
            info_data = await resp.json()
        
        temp_map = {}
        for s in info_data.get('symbols', []):
            if s.get('status') == 'TRADING' and s.get('isSpotTradingAllowed', True):
                quote, base = s.get('quoteAsset'), s.get('baseAsset')
                if quote in ('USDT', 'USDC'):
                    # Extract stepSize for ultra-fast execution formatting later
                    step_size = 1.0
                    for f in s.get('filters', []):
                        if f.get('filterType') == 'LOT_SIZE':
                            step_size = float(f.get('stepSize', '1.0'))
                            break
                    
                    temp_map.setdefault(base, {})[quote] = {
                        'symbol': s.get('symbol'),
                        'step_size': step_size
                    }
        
        _SYMBOL_CACHE = {
            base: pairs for base, pairs in temp_map.items()
            if 'USDT' in pairs and 'USDC' in pairs
        }
    return _SYMBOL_CACHE

async def run_arbitrage_bot(poll_interval=0.2, top_n=10):
    timeout = aiohttp.ClientTimeout(total=5)
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())
    trade_executed = False  
    
    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        print("Initializing active symbol cache & fetching lot sizes...")
        cache = await init_active_symbol_cache(session)
        price_url = "https://api.binance.com/api/v3/ticker/price"
        
        while not trade_executed:
            try:
                timestamp = datetime.now()
                async with session.get(price_url) as resp:
                    if resp.status != 200:
                        await asyncio.sleep(1)
                        continue
                    price_data = await resp.json()
                
                price_map = {item['symbol']: float(item['price']) for item in price_data if float(item['price']) > 0}
                results = []
                
                for base, pairs in cache.items():
                    u_sym_data, c_sym_data = pairs['USDT'], pairs['USDC']
                    u_sym, c_sym = u_sym_data['symbol'], c_sym_data['symbol']
                    
                    if u_sym in price_map and c_sym in price_map:
                        u_price, c_price = price_map[u_sym], price_map[c_sym]
                        
                        spread = ((u_price - c_price) / c_price) * 100
                        
                        # EXECUTION TRIGGER (USDC is cheaper by TARGET_SPREAD_PERCENT)
                        if spread >= TARGET_SPREAD_PERCENT and not trade_executed:
                            clear_output(wait=True)
                            print("="*70)
                            print(f"🚀 ARBITRAGE TRIGGERED! {base}")
                            print(f"USDT Price: {u_price} | USDC Price: {c_price} | Spread: {spread:.3f}%")
                            print("="*70)
                            
                            trade_executed = True # Lock loop
                            
                            # --- LEG 1: BUY ---
                            buy_res, buy_latency = await send_order(
                                symbol_formatted=c_sym,
                                side="BUY",
                                params_extra={"quoteOrderQty": str(TRADE_QUOTE_AMOUNT)},
                                test_mode=TEST_MODE
                            )
                            
                            mode = "TEST" if TEST_MODE else "LIVE"
                            print(f"[{mode} BUY] Pair: {c_sym} | RTT: {buy_latency:.2f} ms")
                            
                            # --- LEG 2: SELL ---
                            executed_qty = 0.0
                            
                            if TEST_MODE:
                                # In test mode, Binance returns an empty result object, so we mock the filled amount
                                print("🛠️ TEST MODE: Mocking order fill...")
                                executed_qty = TRADE_QUOTE_AMOUNT / c_price 
                            else:
                                # In LIVE mode, verify the status and extract the exact quantity received instantly
                                result_data = buy_res.get("result", {})
                                status = result_data.get("status")
                                
                                if status in ["FILLED", "PARTIALLY_FILLED"]:
                                    executed_qty = float(result_data.get("executedQty", 0.0))
                                    print(f"✅ BUY Verified ({status}). Instantly routing SELL order...")
                                else:
                                    print(f"❌ BUY failed or pending. Status: {status}. Sell aborted.")
                                    break
                            
                            if executed_qty > 0:
                                # Truncate exact amount using the cached stepSize (ZERO latency overhead)
                                step_size = u_sym_data['step_size']
                                sell_qty_str = round_step_size(executed_qty, step_size)
                                
                                sell_res, sell_latency = await send_order(
                                    symbol_formatted=u_sym,
                                    side="SELL",
                                    params_extra={"quantity": sell_qty_str},
                                    test_mode=TEST_MODE
                                )
                                print(f"[{mode} SELL] Pair: {u_sym} | Qty: {sell_qty_str} | RTT: {sell_latency:.2f} ms")
                            
                            print("\n✅ Arbitrage execution complete. Shutting down scanner.")
                            break # Exits the for-loop
                        
                        # Only prepare display for meaningful spreads to save CPU overhead
                        if abs(spread) >= 0.1:
                            results.append({
                                'Symbol': base,
                                'USDT_Price': u_price,
                                'USDC_Price': c_price,
                                'Spread_%': spread,
                                'Abs_Spread': abs(spread)
                            })
                
                # Update console view
                if not trade_executed and results:
                    df = pd.DataFrame(results).sort_values(by='Abs_Spread', ascending=False)
                    df = df[['Symbol', 'USDT_Price', 'USDC_Price', 'Spread_%']].head(top_n)
                    df.set_index('Symbol', inplace=True)
                    
                    clear_output(wait=True)
                    print(f"⚡ SCANNER [{timestamp.strftime('%H:%M:%S.%f')[:-3]}] | Target Spread: >{TARGET_SPREAD_PERCENT}% | Status: Watching...")
                    display(df)
                
                if trade_executed:
                    break
                    
                await asyncio.sleep(poll_interval)
                
            except asyncio.CancelledError:
                break
            except Exception as e:
                print(f"[{datetime.now().strftime('%H:%M:%S')}] Loop Error: {e}")
                await asyncio.sleep(1)

# =====================================================================
# START SCRIPT
# =====================================================================
try:
    await run_arbitrage_bot(poll_interval=1.0, top_n=10)
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[Stopped] Live arbitrage stream halted.")

🚀 ARBITRAGE TRIGGERED! XVG
USDT Price: 0.003062 | USDC Price: 0.003045 | Spread: 0.558%
[TEST BUY] Pair: XVGUSDC | RTT: 196.43 ms
🛠️ TEST MODE: Mocking order fill...
[TEST SELL] Pair: XVGUSDT | Qty: 33169 | RTT: 236.03 ms

✅ Arbitrage execution complete. Shutting down scanner.


# C

In [13]:
# =====================================================================
# PRICE CHECK & CONDITIONAL EXECUTION ENGINE
# =====================================================================
async def get_book_ticker(symbol: str):
    """Fetches real-time best bid and best ask prices for a symbol over WebSocket API."""
    ws = await get_ws_connection()
    symbol_formatted, _ = parse_symbol(symbol)
    now = int(time.time() * 1000)

    req = {
        "id": f"ticker_{now}",
        "method": "ticker.bookTicker",
        "params": {"symbol": symbol_formatted},
    }
    await ws.send(json.dumps(req))
    res = json.loads(await ws.recv())

    result = res.get("result", {})
    return {
        "symbol": symbol_formatted,
        "bid": float(result.get("bidPrice", 0.0)),  # Price to sell at
        "ask": float(result.get("askPrice", 0.0)),  # Price to buy at
    }


async def fast_conditional_trade(
    buy_symbol: str,
    sell_symbol: str,
    quote_amount: float = 101.0,
    min_discount_pct: float = 0.5,
    test_mode: bool = False,
):
    """
    Checks if the buying pair's ask price is at least min_discount_pct% lower
    than the selling pair's bid price. If condition is met, buys immediately
    and sells on the target pair right after.
    """
    # 1. Fetch current order book tickers
    buy_ticker = await get_book_ticker(buy_symbol)
    sell_ticker = await get_book_ticker(sell_symbol)

    buy_ask = buy_ticker["ask"]
    sell_bid = sell_ticker["bid"]

    if buy_ask <= 0 or sell_bid <= 0:
        print("⚠️ Execution Aborted: Failed to fetch valid ticker prices.")
        return None

    # 2. Calculate price difference percentage
    # Discount % = (Sell Bid - Buy Ask) / Sell Bid * 100
    actual_discount_pct = ((sell_bid - buy_ask) / sell_bid) * 100

    print(
        f"📊 Price Check | Buy Ask ({buy_ticker['symbol']}): {buy_ask} | "
        f"Sell Bid ({sell_ticker['symbol']}): {sell_bid} | Spread: {actual_discount_pct:.3f}%"
    )

    # 3. Evaluate condition (Buy price must be at least min_discount_pct lower)
    if actual_discount_pct < min_discount_pct:
        print(
            f"⛔ Execution Skipped: Required {min_discount_pct}% discount, but actual is {actual_discount_pct:.3f}%."
        )
        return {"status": "SKIPPED", "reason": "Spread condition not met"}

    print(
        f"✅ Condition Met! ({actual_discount_pct:.3f}% >= {min_discount_pct}%). Executing orders..."
    )

    # 4. Execute Buy Order
    buy_res = await fast_buy(
        symbol=buy_symbol, quote_amount=quote_amount, test_mode=test_mode
    )

    # Check for order submission errors in live mode
    if not test_mode and "error" in buy_res:
        print(f"❌ Buy Failed: {buy_res['error']}. Aborting sell.")
        return buy_res

    # 5. Execute Instant Sell Order
    sell_res = await fast_sell_all(symbol=sell_symbol, test_mode=test_mode)

    return {"buy_response": buy_res, "sell_response": sell_res}


# =====================================================================
# EXAMPLE USAGE
# =====================================================================
# Example: Buy ZKP/USDT if it's 0.5% cheaper than ZKP/USDC bid price, then sell instantly on ZKP/USDC
await fast_conditional_trade(
    buy_symbol="COKIE/USDC",
    sell_symbol="COOKIE/USDT",
    quote_amount=15.0,
    min_discount_pct=0.5,  # 0.5% minimum price difference
    test_mode=True,
)

⚠️ Execution Aborted: Failed to fetch valid ticker prices.


In [15]:
import asyncio
import hashlib
import hmac
import json
import math
import time
from datetime import datetime
from urllib.parse import urlencode

import aiohttp
import pandas as pd
import websockets

# =====================================================================
# CONFIGURATION & CONSTANTS
# =====================================================================
WS_URL = "wss://ws-api.binance.com:443/ws-api/v3"

pd.set_option(
    "display.float_format",
    lambda x: f"{x:.8f}".rstrip("0").rstrip(".") if x != 0 else "0",
)

ws_client = None
_SYMBOL_CACHE = {}
in_flight_trades = set()  # Tracks active trades to prevent duplicate orders


# =====================================================================
# WEBSOCKET & ROUTING ENGINE
# =====================================================================
async def get_ws_connection():
    global ws_client
    is_closed = True
    if ws_client is not None:
        if hasattr(ws_client, "close_code"):
            is_closed = ws_client.close_code is not None
        elif hasattr(ws_client, "closed"):
            is_closed = ws_client.closed

    if ws_client is None or is_closed:
        ws_client = await websockets.connect(WS_URL)
    return ws_client


def generate_signature(params: dict, secret: str) -> str:
    sorted_params = dict(sorted(params.items()))
    query_string = urlencode(sorted_params)
    return hmac.new(
        secret.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256
    ).hexdigest()


def parse_symbol(raw_symbol: str):
    s = raw_symbol.upper().replace("/", "").replace("-", "").replace("_", "")
    quotes = ["USDT", "USDC", "FDUSD", "BUSD", "BTC", "ETH", "BNB", "EUR"]
    base = s
    for q in quotes:
        if s.endswith(q):
            base = s[: -len(q)]
            break
    return s, base


def round_step_size(quantity: float, step_size: float) -> str:
    if step_size <= 0:
        return str(quantity)
    precision = int(round(-math.log10(step_size)))
    if precision <= 0:
        factor = 10 ** abs(precision)
        return f"{int(math.floor(quantity / factor) * factor)}"
    factor = 10**precision
    return f"{(math.floor(quantity * factor) / factor):.{precision}f}"


async def get_book_ticker(symbol: str):
    ws = await get_ws_connection()
    symbol_formatted, _ = parse_symbol(symbol)
    now = int(time.time() * 1000)

    req = {
        "id": f"ticker_{now}",
        "method": "ticker.bookTicker",
        "params": {"symbol": symbol_formatted},
    }
    await ws.send(json.dumps(req))
    res = json.loads(await ws.recv())
    result = res.get("result", {})
    return {
        "symbol": symbol_formatted,
        "bid": float(result.get("bidPrice", 0.0)),
        "ask": float(result.get("askPrice", 0.0)),
    }


async def send_order(
    symbol_formatted: str, side: str, params_extra: dict, test_mode: bool = False
):
    ws = await get_ws_connection()
    now = int(time.time() * 1000)
    params = {
        "apiKey": API_KEY,
        "symbol": symbol_formatted,
        "side": side.upper(),
        "type": "MARKET",
        "newOrderRespType": "ACK",
        "timestamp": now,
        **params_extra,
    }
    params["signature"] = generate_signature(params, API_SECRET)

    payload = {
        "id": f"ord_{now}",
        "method": "order.test" if test_mode else "order.place",
        "params": params,
    }
    t0 = time.perf_counter()
    await ws.send(json.dumps(payload))
    res = json.loads(await ws.recv())
    latency_ms = (time.perf_counter() - t0) * 1000
    return res, latency_ms


async def fast_buy(
    symbol: str, quote_amount: float = 101.0, test_mode: bool = False
):
    symbol_formatted, _ = parse_symbol(symbol)
    res, latency_ms = await send_order(
        symbol_formatted=symbol_formatted,
        side="BUY",
        params_extra={"quoteOrderQty": str(quote_amount)},
        test_mode=test_mode,
    )
    print(
        f"[{'TEST' if test_mode else 'LIVE'} BUY] {symbol_formatted} | Spend: ${quote_amount} | RTT: {latency_ms:.2f}ms"
    )
    return res


async def fast_sell_all(
    symbol: str, min_notional_usdt: float = 5.0, test_mode: bool = False
):
    ws = await get_ws_connection()
    symbol_formatted, base_asset = parse_symbol(symbol)
    now = int(time.time() * 1000)

    # 1. Fetch stepSize filter
    info_req = {
        "id": f"info_{now}",
        "method": "exchangeInfo",
        "params": {"symbol": symbol_formatted},
    }
    await ws.send(json.dumps(info_req))
    info_res = json.loads(await ws.recv())

    step_size = 1.0
    for f in (
        info_res.get("result", {})
        .get("symbols", [{}])[0]
        .get("filters", [])
    ):
        if f.get("filterType") == "LOT_SIZE":
            step_size = float(f.get("stepSize", 1.0))
            break

    # 2. Fetch Account Balance
    bal_params = {"apiKey": API_KEY, "timestamp": now}
    bal_params["signature"] = generate_signature(bal_params, API_SECRET)
    await ws.send(
        json.dumps(
            {
                "id": f"bal_{now}",
                "method": "account.status",
                "params": bal_params,
            }
        )
    )
    bal_res = json.loads(await ws.recv())

    free_bal = 0.0
    for b in bal_res.get("result", {}).get("balances", []):
        if b["asset"] == base_asset:
            free_bal = float(b["free"])
            break

    if free_bal <= 0:
        if test_mode:
            free_bal = 100.0
        else:
            print(f"⚠️ Sell Skipped: Zero balance for {base_asset}")
            return {"error": "No balance"}

    qty_str = round_step_size(free_bal, step_size)
    if float(qty_str) <= 0:
        print(f"⚠️ Sell Skipped: Quantity {free_bal} below stepSize {step_size}")
        return {"error": "Quantity below stepSize"}

    res, latency_ms = await send_order(
        symbol_formatted=symbol_formatted,
        side="SELL",
        params_extra={"quantity": qty_str},
        test_mode=test_mode,
    )
    print(
        f"[{'TEST' if test_mode else 'LIVE'} SELL] {symbol_formatted} | Qty: {qty_str} | RTT: {latency_ms:.2f}ms"
    )
    return res


async def fast_conditional_trade(
    buy_symbol: str,
    sell_symbol: str,
    quote_amount: float = 101.0,
    min_discount_pct: float = 0.5,
    test_mode: bool = False,
):
    buy_ticker = await get_book_ticker(buy_symbol)
    sell_ticker = await get_book_ticker(sell_symbol)

    buy_ask = buy_ticker["ask"]
    sell_bid = sell_ticker["bid"]

    if buy_ask <= 0 or sell_bid <= 0:
        return {"status": "FAILED", "reason": "Invalid tickers"}

    actual_discount_pct = ((sell_bid - buy_ask) / sell_bid) * 100

    print(
        f"\n🎯 [WS Check] Buy Ask ({buy_ticker['symbol']}): {buy_ask} | "
        f"Sell Bid ({sell_ticker['symbol']}): {sell_bid} | Spread: {actual_discount_pct:.3f}%"
    )

    if actual_discount_pct < min_discount_pct:
        print(
            f"⛔ Skipped: Book spread ({actual_discount_pct:.3f}%) < Threshold ({min_discount_pct}%)"
        )
        return {"status": "SKIPPED", "reason": "Spread condition not met"}

    print(f"⚡ Order Book Verified! Executing trade sequence...")
    buy_res = await fast_buy(
        symbol=buy_symbol, quote_amount=quote_amount, test_mode=test_mode
    )

    if not test_mode and "error" in buy_res:
        print(f"❌ Buy Failed: {buy_res['error']}. Halting sell.")
        return buy_res

    sell_res = await fast_sell_all(symbol=sell_symbol, test_mode=test_mode)
    return {"buy_response": buy_res, "sell_response": sell_res}


# =====================================================================
# INTEGRATED SCANNER + AUTOMATED EXECUTOR
# =====================================================================
async def init_active_symbol_cache(session):
    global _SYMBOL_CACHE
    if not _SYMBOL_CACHE:
        info_url = "https://api.binance.com/api/v3/exchangeInfo?permissions=SPOT"
        async with session.get(info_url) as resp:
            if resp.status != 200:
                raise RuntimeError(
                    f"Failed to fetch exchange info: HTTP {resp.status}"
                )
            info_data = await resp.json()

        temp_map = {}
        for s in info_data.get("symbols", []):
            if s.get("status") == "TRADING" and s.get(
                "isSpotTradingAllowed", True
            ):
                quote = s.get("quoteAsset")
                base = s.get("baseAsset")
                if quote in ("USDT", "USDC"):
                    temp_map.setdefault(base, {})[quote] = s.get("symbol")

        _SYMBOL_CACHE = {
            base: pairs
            for base, pairs in temp_map.items()
            if "USDT" in pairs and "USDC" in pairs
        }
    return _SYMBOL_CACHE


async def process_trade_opportunity(
    base_asset: str,
    spread_pct: float,
    quote_amount: float,
    min_spread_percent: float,
    test_mode: bool,
):
    """Determines buy/sell pair directions and triggers execution async."""
    if base_asset in in_flight_trades:
        return  # Skip if a trade for this symbol is currently executing

    in_flight_trades.add(base_asset)
    try:
        # Route pair direction based on price differential
        if spread_pct > 0:
            # USDT is higher than USDC -> Buy USDC pair, Sell USDT pair
            buy_symbol = f"{base_asset}/USDC"
            sell_symbol = f"{base_asset}/USDT"
        else:
            # USDC is higher than USDT -> Buy USDT pair, Sell USDC pair
            buy_symbol = f"{base_asset}/USDT"
            sell_symbol = f"{base_asset}/USDC"

        await fast_conditional_trade(
            buy_symbol=buy_symbol,
            sell_symbol=sell_symbol,
            quote_amount=quote_amount,
            min_discount_pct=min_spread_percent,
            test_mode=test_mode,
        )
    finally:
        in_flight_trades.remove(base_asset)


async def run_clean_realtime_arbitrage(
    poll_interval=0.2,
    min_spread_percent=0.5,
    quote_amount=101.0,
    top_n=15,
    test_mode=True,
):
    timeout = aiohttp.ClientTimeout(total=5)
    connector = aiohttp.TCPConnector(resolver=aiohttp.ThreadedResolver())

    async with aiohttp.ClientSession(
        connector=connector, timeout=timeout
    ) as session:
        print("Initializing active symbol cache...")
        cache = await init_active_symbol_cache(session)
        price_url = "https://api.binance.com/api/v3/ticker/price"

        while True:
            try:
                timestamp = datetime.now()
                async with session.get(price_url) as resp:
                    if resp.status != 200:
                        await asyncio.sleep(1)
                        continue
                    price_data = await resp.json()

                price_map = {
                    item["symbol"]: float(item["price"])
                    for item in price_data
                    if float(item["price"]) > 0
                }

                results = []
                for base, pairs in cache.items():
                    u_sym, c_sym = pairs["USDT"], pairs["USDC"]
                    if u_sym in price_map and c_sym in price_map:
                        u_price, c_price = price_map[u_sym], price_map[c_sym]
                        if u_price <= 0 or c_price <= 0:
                            continue

                        spread = ((u_price - c_price) / c_price) * 100

                        if abs(spread) >= min_spread_percent:
                            results.append(
                                {
                                    "Symbol": base,
                                    "USDT_Price": u_price,
                                    "USDC_Price": c_price,
                                    "Spread_%": spread,
                                    "Abs_Spread": abs(spread),
                                }
                            )

                if results:
                    df = pd.DataFrame(results)
                    df.sort_values(
                        by="Abs_Spread", ascending=False, inplace=True
                    )
                    df_display = df[
                        ["Symbol", "USDT_Price", "USDC_Price", "Spread_%"]
                    ].set_index("Symbol")

                    if top_n:
                        df_display = df_display.head(top_n)

                    from IPython.display import clear_output, display

                    clear_output(wait=True)
                    print(
                        f"⚡ REAL-TIME ARBITRAGE SCANNER [{timestamp.strftime('%H:%M:%S.%f')[:-3]}] | Active Dual Pairs: {len(df)}"
                    )
                    display(df_display)

                    # Trigger trade on the top arbitrage candidate
                    top_candidate = df.iloc[0]
                    asyncio.create_task(
                        process_trade_opportunity(
                            base_asset=top_candidate["Symbol"],
                            spread_pct=top_candidate["Spread_%"],
                            quote_amount=quote_amount,
                            min_spread_percent=min_spread_percent,
                            test_mode=test_mode,
                        )
                    )

                await asyncio.sleep(poll_interval)

            except asyncio.CancelledError:
                break
            except Exception as e:
                print(
                    f"[{datetime.now().strftime('%H:%M:%S')}] Loop Error: {e}"
                )
                await asyncio.sleep(1)


# =====================================================================
# RUN INTEGRATED SCANNER & EXECUTOR
# =====================================================================
try:
    await run_clean_realtime_arbitrage(
        poll_interval=0.5,
        min_spread_percent=0.5,  # Minimum 0.5% spread
        quote_amount=15.0,  # Spend $101 per trade
        top_n=10,
        test_mode=True,  # Set False for real orders
    )
except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[Stopped] Scanner and trader halted.")

⚡ REAL-TIME ARBITRAGE SCANNER [13:36:59.884] | Active Dual Pairs: 10


,Original_Index,USDT_Price,USDC_Price,Spread_%
1,CHIP,0.05753,0.05712,0.71778711
2,HUMA,0.02185,0.022,-0.68181818
3,BLUR,0.01669,0.0168,-0.6547619
4,SIGN,0.00998,0.01004,-0.59760956
5,TLM,0.001458,0.00145,0.55172414
6,HEI,0.1483,0.1475,0.54237288
7,MET,0.1898,0.1888,0.52966102
8,NOM,0.00192,0.00193,-0.51813472
9,UMA,0.389,0.391,-0.51150895
10,ACE,0.1769,0.1778,-0.50618673


In [12]:
# Execution Conditions
import asyncio
import hashlib
import hmac
import json
import time
from urllib.parse import urlencode
import websockets

# =====================================================================
# CONFIGURATION
# =====================================================================
WS_URL = "wss://ws-api.binance.com:443/ws-api/v3"

# Persistent WebSocket client connection across notebook cells
ws_client = None


# =====================================================================
# WEBSOCKET & AUTH UTILITIES
# =====================================================================
async def get_ws_connection():
    """Maintains a persistent socket connection compatible with websockets v10 through v14+."""
    global ws_client

    is_closed = True
    if ws_client is not None:
        if hasattr(ws_client, "close_code"):
            is_closed = ws_client.close_code is not None  # websockets >= 13.0
        elif hasattr(ws_client, "closed"):
            is_closed = ws_client.closed  # websockets < 13.0

    if ws_client is None or is_closed:
        ws_client = await websockets.connect(WS_URL)

    return ws_client


def generate_signature(params: dict, secret: str) -> str:
    """Generates HMAC-SHA256 signature sorted alphabetically according to Binance API rules."""
    sorted_params = dict(sorted(params.items()))
    query_string = urlencode(sorted_params)
    return hmac.new(
        secret.encode("utf-8"), query_string.encode("utf-8"), hashlib.sha256
    ).hexdigest()


def parse_symbol(raw_symbol: str):
    """Normalizes symbols like 'ZKP/USDT', 'sol-usdc', or 'BTCUSDT' into Binance formats and extracts the base asset."""
    s = raw_symbol.upper().replace("/", "").replace("-", "").replace("_", "")
    quotes = ["USDT", "USDC", "FDUSD", "BUSD", "BTC", "ETH", "BNB", "EUR"]
    base = s
    for q in quotes:
        if s.endswith(q):
            base = s[: -len(q)]
            break
    return s, base


# =====================================================================
# ORDER ROUTING ENGINE
# =====================================================================
async def send_order(
    symbol_formatted: str, side: str, params_extra: dict, test_mode: bool = False
):
    """Dispatches order.place or order.test over WebSocket frame and measures RTT latency."""
    ws = await get_ws_connection()
    now = int(time.time() * 1000)

    params = {
        "apiKey": API_KEY,
        "symbol": symbol_formatted,
        "side": side.upper(),
        "type": "MARKET",
        "newOrderRespType": "ACK",  # Instant receipt ack without match engine waiting
        "timestamp": now,
        **params_extra,
    }
    params["signature"] = generate_signature(params, API_SECRET)

    method = "order.test" if test_mode else "order.place"
    payload = {
        "id": f"ord_{now}",
        "method": method,
        "params": params,
    }

    t0 = time.perf_counter()
    await ws.send(json.dumps(payload))
    res = json.loads(await ws.recv())
    latency_ms = (time.perf_counter() - t0) * 1000

    return res, latency_ms


async def fast_buy(
    symbol: str, quote_amount: float = 101.0, test_mode: bool = False
):
    """Executes a market buy order using exact quote currency ($101) without pre-fetching order books."""
    symbol_formatted, _ = parse_symbol(symbol)
    res, latency_ms = await send_order(
        symbol_formatted=symbol_formatted,
        side="BUY",
        params_extra={"quoteOrderQty": str(quote_amount)},
        test_mode=test_mode,
    )
    mode_tag = "TEST" if test_mode else "LIVE"
    print(
        f"[{mode_tag} BUY] Pair: {symbol_formatted} | Spend: ${quote_amount} | RTT: {latency_ms:.2f} ms"
    )
    return res


import math


def round_step_size(quantity: float, step_size: float) -> str:
    """Truncates quantity down to the nearest valid stepSize increment to prevent LOT_SIZE errors."""
    if step_size <= 0:
        return str(quantity)

    # Determine decimal precision required by step_size
    precision = int(round(-math.log10(step_size)))
    if precision <= 0:
        factor = 10 ** abs(precision)
        truncated = math.floor(quantity / factor) * factor
        return f"{int(truncated)}"

    factor = 10**precision
    truncated = math.floor(quantity * factor) / factor
    return f"{truncated:.{precision}f}"


async def fast_sell_all(
    symbol: str, min_notional_usdt: float = 5.0, test_mode: bool = False
):
    """Fetches available balance, applies LOT_SIZE stepSize filter, and market sells 100% of holdings."""
    ws = await get_ws_connection()
    symbol_formatted, base_asset = parse_symbol(symbol)
    now = int(time.time() * 1000)

    # 1. Fetch Symbol Info to get LOT_SIZE stepSize filter
    info_payload = {
        "id": f"info_{now}",
        "method": "exchangeInfo",
        "params": {"symbol": symbol_formatted},
    }
    await ws.send(json.dumps(info_payload))
    info_res = json.loads(await ws.recv())

    step_size = 1.0
    filters = (
        info_res.get("result", {})
        .get("symbols", [{}])[0]
        .get("filters", [])
    )
    for f in filters:
        if f.get("filterType") == "LOT_SIZE":
            step_size = float(f.get("stepSize", 1.0))
            break

    # 2. Fetch Account Balance
    bal_params = {"apiKey": API_KEY, "timestamp": now}
    bal_params["signature"] = generate_signature(bal_params, API_SECRET)

    await ws.send(
        json.dumps(
            {
                "id": f"bal_{now}",
                "method": "account.status",
                "params": bal_params,
            }
        )
    )
    bal_res = json.loads(await ws.recv())

    free_bal = 0.0
    for b in bal_res.get("result", {}).get("balances", []):
        if b["asset"] == base_asset:
            free_bal = float(b["free"])
            break

    if free_bal <= 0:
        if test_mode:
            free_bal = 100.0  # Mock quantity for testing
        else:
            print(f"⚠️ Sell Skipped: No available balance for {base_asset}")
            return {"error": f"No balance for {base_asset}"}

    # 3. Apply LOT_SIZE stepSize precision truncation
    qty_str = round_step_size(free_bal, step_size)

    if float(qty_str) <= 0:
        print(
            f"⚠️ Sell Skipped: Available balance ({free_bal}) is smaller than stepSize ({step_size})."
        )
        return {"error": "Quantity below stepSize minimum"}

    # 4. Dispatch Order
    res, latency_ms = await send_order(
        symbol_formatted=symbol_formatted,
        side="SELL",
        params_extra={"quantity": qty_str},
        test_mode=test_mode,
    )

    mode_tag = "TEST" if test_mode else "LIVE"
    print(
        f"[{mode_tag} SELL] Symbol: {symbol_formatted} | Validated Qty: {qty_str} | RTT: {latency_ms:.2f} ms"
    )
    return res
    
    
# Dry-run test $101 buy on ZKP/USDT
await fast_buy("ZKP/USDT", quote_amount=101.0, test_mode=True)

# Dry-run test 100% sell on ZKP/USDT
await fast_sell_all("ZKP/USDT", test_mode=True)

[TEST BUY] Pair: ZKPUSDT | Spend: $101.0 | RTT: 232.78 ms
[TEST SELL] Symbol: ZKPUSDT | Validated Qty: 100.0 | RTT: 188.19 ms


{'id': 'ord_1788595524794',
 'status': 400,
 'error': {'code': -1013, 'msg': 'Filter failure: NOTIONAL'},
 'rateLimits': [{'rateLimitType': 'REQUEST_WEIGHT',
   'interval': 'MINUTE',
   'intervalNum': 1,
   'limit': 6000,
   'count': 44}]}

In [15]:
# Dry-run test $101 buy on ZKP/USDT
await fast_buy("ZKP/USDT", quote_amount=101.0, test_mode=True)

# Dry-run test 100% sell on ZKP/USDT
await fast_sell_all("ZKP/USDT", test_mode=True)

[TEST BUY] Pair: ZKPUSDT | Spend: $101.0 | RTT: 229.11 ms
[TEST SELL] Symbol: ZKPUSDT | Validated Qty: 100.0 | RTT: 228.31 ms


{'id': 'ord_1788589445572',
 'status': 400,
 'error': {'code': -1013, 'msg': 'Filter failure: NOTIONAL'},
 'rateLimits': [{'rateLimitType': 'REQUEST_WEIGHT',
   'interval': 'MINUTE',
   'intervalNum': 1,
   'limit': 6000,
   'count': 70}]}